# Workflow Planning Evaluator — Bug Bash Notebook

This notebook walks you through:
1. Installing required packages
2. Setting up tracing with Application Insights
3. Running 3 multi-agent HR recruitment workflows:
   - **Group Chat** with agent manager (orchestrator selects speakers)
   - **Magentic** multi-source sourcing with compliance
   - **Sequential + Human-in-the-Loop** hiring pipeline
4. Fetching traces from App Insights and converting them
5. Running the Workflow Planning Evaluator on each trace

**Prerequisites:**
- Azure AI project with Application Insights attached
- Run `az login` before starting
- Set your environment variables in the cell below

## 1. Install Packages

In [ ]:
# Install agent framework
%pip install agent-framework

# Install azure-ai-evaluation in editable mode (run from the azure-sdk-for-python repo root)
%pip install -e ../../../../../../..[redteam]

# Additional dependencies for tracing
%pip install azure-monitor-opentelemetry azure-monitor-query azure-identity opentelemetry-api python-dotenv

## 2. Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# ── Set these environment variables (or use a .env file) ──
# os.environ["AZURE_AI_PROJECT_ENDPOINT"] = "https://<your-project>.services.ai.azure.com/api"
# os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"] = "gpt-4o"
# os.environ["APP_INSIGHTS_WORKSPACE_ID"] = "<your-workspace-id>"

assert os.environ.get("AZURE_AI_PROJECT_ENDPOINT"), "Set AZURE_AI_PROJECT_ENDPOINT"
assert os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME"), "Set AZURE_AI_MODEL_DEPLOYMENT_NAME"
assert os.environ.get("APP_INSIGHTS_WORKSPACE_ID"), "Set APP_INSIGHTS_WORKSPACE_ID"

print("Environment configured!")
print(f"  Project endpoint: {os.environ['AZURE_AI_PROJECT_ENDPOINT']}")
print(f"  Model deployment: {os.environ['AZURE_AI_MODEL_DEPLOYMENT_NAME']}")
print(f"  Workspace ID:     {os.environ['APP_INSIGHTS_WORKSPACE_ID'][:8]}...")

## 3. Tracing Setup

Configure Azure Monitor telemetry and enable instrumentation with sensitive data capture.

In [ ]:
from agent_framework.azure import AzureAIClient
from agent_framework.observability import enable_instrumentation, get_tracer
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import AzureCliCredential
from opentelemetry.trace import SpanKind
from opentelemetry.trace.span import format_trace_id

credential = AzureCliCredential()
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=credential,
)
client = AzureAIClient(project_client=project_client)

# Configure Azure Monitor and enable instrumentation
await client.configure_azure_monitor(enable_live_metrics=True)
enable_instrumentation(enable_sensitive_data=True)

tracer = get_tracer()
print("Tracing configured with App Insights + sensitive data enabled!")

In [ ]:
# We'll collect trace IDs from each workflow run
collected_trace_ids = {}

## 4. Import HR Agents & Tools

The HR mock data, function tools, and agent factories are defined in `scripts/hr_tools.py` and `scripts/hr_agents.py`.

In [ ]:
import sys
sys.path.insert(0, ".")

from scripts.hr_agents import (
    create_compliance_guard,
    create_evaluator,
    create_magentic_manager,
    create_mobility_scout,
    create_orchestrator,
    create_req_master,
    create_scheduler,
    create_talent_scout,
)

print("HR agents and tools loaded!")

## 5. Workflow 1: Group Chat with Agent Manager

An HR group chat where an **Orchestrator agent** decides which specialist speaks next.
The team works together to find candidates for a Senior Software Engineer position.

**Feel free to modify the agents, task, and termination condition!**

In [ ]:
from typing import cast
from agent_framework import Agent, AgentResponseUpdate, Message, WorkflowEvent
from agent_framework.orchestrations import GroupChatBuilder

# ── Create HR agents (modify these!) ──
orchestrator = create_orchestrator(client)
req_master = create_req_master(client)
talent_scout = create_talent_scout(client)
evaluator = create_evaluator(client)

# ── Build the group chat workflow ──
group_chat_workflow = (
    GroupChatBuilder(
        participants=[req_master, talent_scout, evaluator],
        termination_condition=lambda messages: sum(1 for m in messages if m.role == "assistant") >= 6,
        intermediate_outputs=True,
        orchestrator_agent=orchestrator,
    )
    .with_termination_condition(lambda messages: sum(1 for m in messages if m.role == "assistant") >= 6)
    .build()
)

# ── Run with tracing ──
task = (
    "Help me fill the Senior Software Engineer position JOB-SWE-2025-001. "
    "Find candidates, evaluate them, and recommend a shortlist."
)

print(f"\nTask: {task}\n" + "=" * 80)

with tracer.start_as_current_span("group_chat_workflow", kind=SpanKind.CLIENT) as span:
    trace_id = format_trace_id(span.get_span_context().trace_id)
    collected_trace_ids["group_chat"] = trace_id
    print(f"Trace ID: {trace_id}\n")

    last_response_id = None
    async for event in group_chat_workflow.run(task, stream=True):
        if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
            rid = event.data.response_id
            if rid != last_response_id:
                if last_response_id is not None:
                    print("\n")
                print(f"{event.data.author_name}:", end=" ", flush=True)
                last_response_id = rid
            print(event.data.text, end="", flush=True)
        elif event.type == "output":
            outputs = cast(list[Message], event.data)
            print("\n" + "=" * 80)
            print("\nFinal Conversation:")
            for message in outputs:
                print(f"  {message.author_name or message.role}: {message.text[:200]}...\n")

print(f"\n\u2705 Group Chat complete! Trace ID: {collected_trace_ids['group_chat']}")

## 6. Workflow 2: Magentic Multi-Source Sourcing

A Magentic orchestration where a manager coordinates multiple specialists to source
candidates from all channels, evaluate them, and run compliance checks.

**Feel free to modify the agents, task, and configuration!**

In [ ]:
import json as json_module
from agent_framework.orchestrations import GroupChatRequestSentEvent, MagenticBuilder, MagenticProgressLedger

# ── Create HR agents (modify these!) ──
magentic_manager = create_magentic_manager(client)
mag_talent_scout = create_talent_scout(client)
mag_mobility_scout = create_mobility_scout(client)
mag_evaluator = create_evaluator(client)
mag_compliance = create_compliance_guard(client)

# ── Build the Magentic workflow ──
magentic_workflow = MagenticBuilder(
    participants=[mag_talent_scout, mag_mobility_scout, mag_evaluator, mag_compliance],
    intermediate_outputs=True,
    manager_agent=magentic_manager,
    max_round_count=10,
    max_stall_count=3,
    max_reset_count=2,
).build()

# ── Run with tracing ──
task = (
    "We need to hire for JOB-SWE-2025-001 (Senior Software Engineer). "
    "Source candidates from all channels (external job boards and internal transfers), "
    "evaluate them, and run a compliance check on the final shortlist."
)

print(f"\nTask: {task}\n" + "=" * 80)

with tracer.start_as_current_span("magentic_workflow", kind=SpanKind.CLIENT) as span:
    trace_id = format_trace_id(span.get_span_context().trace_id)
    collected_trace_ids["magentic"] = trace_id
    print(f"Trace ID: {trace_id}\n")

    last_response_id = None
    output_event = None
    async for event in magentic_workflow.run(task, stream=True):
        if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
            response_id = event.data.response_id
            if response_id != last_response_id:
                if last_response_id is not None:
                    print("\n")
                print(f"- {event.executor_id}:", end=" ", flush=True)
                last_response_id = response_id
            print(event.data, end="", flush=True)
        elif event.type == "magentic_orchestrator":
            print(f"\n[Magentic Orchestrator] Type: {event.data.event_type.name}")
            if isinstance(event.data.content, Message):
                print(f"Plan:\n{event.data.content.text[:300]}...")
            elif isinstance(event.data.content, MagenticProgressLedger):
                print(f"Progress:\n{json_module.dumps(event.data.content.to_dict(), indent=2)[:300]}...")
        elif event.type == "output":
            output_event = event

    if output_event:
        outputs = cast(list[Message], output_event.data)
        print("\n" + "=" * 80)
        print("\nFinal Conversation:")
        for message in outputs:
            print(f"  {message.author_name or message.role}: {message.text[:200]}...\n")

print(f"\n\u2705 Magentic complete! Trace ID: {collected_trace_ids['magentic']}")

## 7. Workflow 3: Sequential HR Pipeline with Human-in-the-Loop

A sequential hiring pipeline: ReqMaster \u2192 TalentScout \u2192 Evaluator \u2192 Scheduler.
The workflow **pauses after the Evaluator** so you can review the shortlist before
scheduling interviews.

**When prompted, provide feedback or type 'skip' to approve the shortlist.**

In [ ]:
from collections.abc import AsyncIterable
from agent_framework import AgentExecutorResponse
from agent_framework.orchestrations import AgentRequestInfoResponse, SequentialBuilder


async def process_event_stream(stream: AsyncIterable[WorkflowEvent]):
    """Process events from the workflow stream, handling HITL requests."""
    requests = {}
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, AgentExecutorResponse):
            requests[event.request_id] = event.data
        elif event.type == "output":
            print("\n" + "=" * 60)
            print("WORKFLOW COMPLETE")
            print("=" * 60)
            outputs = cast(list[Message], event.data)
            for message in outputs:
                print(f"  [{message.author_name or message.role}]: {message.text[:300]}...")

    responses = {}
    if requests:
        for request_id, request in requests.items():
            print("\n" + "-" * 40)
            print(f"HUMAN REVIEW REQUESTED")
            print(f"Agent '{request.executor_id}' produced a shortlist:")
            print(f"  {request.agent_response.text[:500]}...")
            print("-" * 40)

            user_input = input("Your feedback (or 'skip' to approve): ")
            if user_input.lower() == "skip":
                responses[request_id] = AgentRequestInfoResponse.approve()
            else:
                responses[request_id] = AgentRequestInfoResponse.from_strings([user_input])

    return responses if responses else None


# ── Create HR agents (modify these!) ──
seq_req_master = create_req_master(client)
seq_talent_scout = create_talent_scout(client)
seq_evaluator = create_evaluator(client)
seq_scheduler = create_scheduler(client)

# ── Build the sequential workflow with HITL ──
sequential_workflow = (
    SequentialBuilder(participants=[seq_req_master, seq_talent_scout, seq_evaluator, seq_scheduler])
    .with_request_info(agents=["Evaluator"])  # Pause after evaluator for human review
    .build()
)

# ── Run with tracing ──
task = (
    "Process the hiring pipeline for JOB-SWE-2025-001: gather requirements, "
    "source candidates, evaluate them (I'll review the shortlist), "
    "then schedule interviews for approved candidates."
)

print(f"\nTask: {task}\n" + "=" * 80)

with tracer.start_as_current_span("sequential_hitl_workflow", kind=SpanKind.CLIENT) as span:
    trace_id = format_trace_id(span.get_span_context().trace_id)
    collected_trace_ids["sequential_hitl"] = trace_id
    print(f"Trace ID: {trace_id}\n")

    stream = sequential_workflow.run(task, stream=True)
    pending_responses = await process_event_stream(stream)
    while pending_responses is not None:
        stream = sequential_workflow.run(stream=True, responses=pending_responses)
        pending_responses = await process_event_stream(stream)

print(f"\n\u2705 Sequential HITL complete! Trace ID: {collected_trace_ids['sequential_hitl']}")

## 8. Summary of Collected Trace IDs

In [ ]:
print("Collected Trace IDs:")
for name, tid in collected_trace_ids.items():
    print(f"  {name}: {tid}")

all_trace_ids = list(collected_trace_ids.values())
print(f"\nTotal: {len(all_trace_ids)} traces")

## 9. Wait for App Insights Ingestion

Application Insights has ~90 second ingestion latency. We wait before querying.

In [ ]:
import time

WAIT_SECONDS = 90
print(f"Waiting {WAIT_SECONDS} seconds for App Insights ingestion...")
for i in range(WAIT_SECONDS, 0, -10):
    print(f"  {i}s remaining...", flush=True)
    time.sleep(min(10, i))
print("Done waiting!")

## 10. Fetch & Convert Traces

Query App Insights for each trace ID, then convert to the structured format expected by the evaluator.

In [ ]:
from scripts.trace_query import query_traces, process_workflow_trace_rows
from scripts.workflow_trace_converter import convert_workflow_traces

workspace_id = os.environ["APP_INSIGHTS_WORKSPACE_ID"]
converted_traces = {}

for workflow_name, trace_id in collected_trace_ids.items():
    print(f"\n{'=' * 60}")
    print(f"Fetching traces for: {workflow_name} (trace_id: {trace_id})")
    print(f"{'=' * 60}")

    try:
        raw_rows = query_traces(
            workspace_id=workspace_id,
            trace_ids=[trace_id],
            lookback_hours=1,
        )
        print(f"  Retrieved {len(raw_rows)} raw rows")

        spans = process_workflow_trace_rows(raw_rows)
        print(f"  Processed into {len(spans)} spans")

        converted = convert_workflow_traces(spans)
        converted_traces[workflow_name] = converted

        print(f"  Workflow type: {converted.get('workflow_type', 'N/A')}")
        print(f"  Agents: {list(converted.get('agents', {}).keys())}")
        print(f"  Invocations: {len(converted.get('invocations', []))}")
        print(f"  Parse failed: {converted.get('parse_failed', False)}")
        print(f"  \u2705 Conversion successful!")
    except Exception as e:
        print(f"  \u274c Error: {e}")
        converted_traces[workflow_name] = None

print(f"\n\nSuccessfully converted: {sum(1 for v in converted_traces.values() if v is not None)}/{len(converted_traces)}")

## 11. (Optional) Inspect Converted Traces

View the converted trace structure for debugging.

In [ ]:
import json

for workflow_name, trace_data in converted_traces.items():
    if trace_data is None:
        continue
    print(f"\n{'=' * 60}")
    print(f"{workflow_name} \u2014 Converted Trace")
    print(f"{'=' * 60}")
    print(json.dumps(trace_data, indent=2, default=str)[:3000])
    print("...")

## 12. Run Workflow Planning Evaluator

Run the evaluator on each converted trace and display results.

In [ ]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.ai.evaluation._evaluators._workflow_planning import _WorkflowPlanningEvaluator

# Configure the model for the evaluator
model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    azure_deployment=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
)

evaluator = _WorkflowPlanningEvaluator(model_config=model_config)

results = {}
for workflow_name, trace_data in converted_traces.items():
    if trace_data is None:
        print(f"\n\u23ed\ufe0f  Skipping {workflow_name} (no trace data)")
        continue

    print(f"\n{'=' * 60}")
    print(f"Evaluating: {workflow_name}")
    print(f"{'=' * 60}")

    try:
        result = evaluator(workflow_trace=trace_data)
        results[workflow_name] = result

        print(f"  Score:  {result.get('workflow_planning', 'N/A')}")
        print(f"  Result: {result.get('workflow_planning_result', 'N/A')}")
        print(f"  Reason: {result.get('workflow_planning_reason', 'N/A')[:300]}")
        print(f"  \u2705 Evaluation complete!")
    except Exception as e:
        print(f"  \u274c Error: {e}")
        results[workflow_name] = {"error": str(e)}

## 13. Results Summary

In [ ]:
print("\n" + "=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)

for workflow_name, result in results.items():
    if "error" in result:
        print(f"\n  {workflow_name}: \u274c ERROR \u2014 {result['error'][:100]}")
    else:
        score = result.get('workflow_planning', 'N/A')
        status = result.get('workflow_planning_result', 'N/A')
        emoji = "\u2705" if status == "pass" else "\u274c"
        print(f"\n  {workflow_name}: {emoji} {status} (score: {score})")
        print(f"    Reason: {result.get('workflow_planning_reason', 'N/A')[:200]}")
        details = result.get('workflow_planning_details', '')
        if details:
            print(f"    Details: {str(details)[:300]}")

print("\n" + "=" * 80)

## Cleanup

In [ ]:
await credential.close()
await project_client.close()
print("Cleaned up!")